# ARCHITECTURE — garment-smoothing pipeline (read me first)

This is the top-level map of the whole system: how a crumpled cloth becomes a robot move, which
files do what, and where to edit when you want to change something. It is written to be understood
**without an LLM in the loop** — by you, or by a local code model with limited context. If you only
ever read one doc, read this one; the others (`HEAD.md`, `STUDENT_RL.md`) are deeper dives this links
to.

> **The whole project in one sentence:** a single Franka arm learns to smooth a crumpled garment by
> (1) a *frozen* perception net that labels every observed point with its place on the flat garment
> pattern, then (2) a *learned* transformer policy that picks one point to grab and a path to drag
> it, trained first by imitation of human VR demos and then by reinforcement against a flatness
> reward.

---

## 0. How to read this

The system splits cleanly into **two halves that are trained separately and never share gradients**:

- **PERCEPTION** — the UV Mapper. Frozen. Turns geometry into *meaning*.
- **POLICY** — the StudentVLA "head". Learned. Turns *meaning* into *action*.

They meet at exactly one place: a per-point feature tensor `(N, 9)`. Hold that boundary in your head
and everything else is plumbing around it.

The genuinely hard, novel code is **two files, ~380 lines total**:
`model/uv_mapper.py` and `model/student_vla.py`. Everything else captures a cloud, runs a subprocess,
writes a file, or moves the arm. Read those two files closely; skim the rest.

---

## 1. The 5-stage pipeline

```
        ┌──── PERCEPTION (frozen) ────┐   ┌──────── POLICY (learned) ────────┐
cloth ─► point cloud ─► UV Mapper ──► state (N,9) ─► StudentVLA ─► grasp+drag ─► robot executes
 (sim     (capture,      (per-point     (featurize:    (transformer    (1 grab idx +     │
  mesh)    FPS→4096)      u,v labels)    join geom+uv)   policy+critic)   variable path)  │
                                                                                          ▼
                                                              _flat_reward(Φ) ◄─ measure flatness
                                                                                          │
                                                                  PPO+GAE  ◄─ learn from ΔΦ over a
                                                                              whole smoothing sequence
```

| # | Stage | Where | One-line job |
|---|-------|-------|--------------|
| 1 | Perception | `model/uv_mapper.py` | per-point `(u,v)` = "which part of the flat garment is this point" |
| 2 | Capture | `head_RL.py:_capture_rl_state` | sim mesh → 4096-pt cloud + normals + centroid |
| 3 | The head | `model/student_vla.py` | cloud → one grasp (point index) + one drag (path) + value |
| 4 | Execute + reward | `head_RL.py:_execute_drag_path`, `_flat_reward` | run the move, score flatness Φ |
| 5 | Learn | `workers/train_student.py` (BC), `workers/student_update.py` (RL) | update the head |

---

## 2. The two conda environments (why there are subprocesses everywhere)

Isaac Sim and the ML stack don't share a Python env, so the system spans **two processes**:

| Env | Runs | Contains |
|-----|------|----------|
| **`fold`** | `head_RL.py` (the sim driver) | Isaac Sim 6.0, Newton cloth, the Franka arm, capture, execution, reward, the RL loop |
| **`infer`** | the `workers/*.py` scripts | PyTorch, the UV Mapper, the StudentVLA |

`head_RL.py` (in `fold`) calls the workers (in `infer`) by shelling out:
`conda run -n infer python workers/<x>.py --npz ... --out ...`. They communicate through **files in
`/tmp`** (an `.npz` state in, a `.json` action out). That is why you see `_run_student_infer`,
`_run_student_update`, etc. — each is one cross-env subprocess call. The `--infer-env` flag
(default `infer`) names the worker env.

---

## 3. CORE FILE 1 — `model/uv_mapper.py` (perception, frozen)

**Problem it solves.** A crumpled garment as a point cloud is *anonymous* — you can't tell collar
from cuff by geometry alone. The UV Mapper predicts, for every point, its coordinate `(u,v) ∈ [0,1]²`
on the garment's flat 2D sewing pattern. That is **correspondence**: anonymous 3D blob → semantically
labelled cloud. It is the "targeting" half of the project (once you know a point's UV, you know where
it *should* go on a flat table).

**I/O** (`forward`, `uv_mapper.py:173`):
```
in : (B, N, 7) = [x, y, z, z_from_table, nx, ny, nz]
out: phi_u (B, N, 128), phi_v (B, N, 128)     # 128-way classification per axis (K=128)
```

**Why classification, not regression** (the key idea): a crumpled point can plausibly belong to two
UV places (front/back overlap). Regressing one `(u,v)` averages them → a wrong middle point. A
softmax over 128 bins stays *multimodal*; `predict_uv` (line 180) takes `argmax` per axis →
`u = ku/127`, plus a confidence = peak softmax probability.

**Architecture** (`encode`, line 164) — three feature sources summed, then a transformer:
```python
f = self.encoder(xyz) + self.pos_emb(coords) + self.normal_emb(normals)
f = self.transformer(f)          # 6-layer, 6-head, d=384, pre-LN self-attention
return self.mlp(f)               # then head_u / head_v → 128-bin logits
```
1. `ResNet3DEncoder` (line 44): voxelize the cloud (120×120×60 @ 1 cm), **sparse 3D convs** (spconv,
   only on occupied voxels), then trilinear-interpolate volume features back onto each point.
2. `SinusoidalPosEmb` (line 24): positional embedding on `[x,y,z,z_from_table]` — the sharp position
   the voxel grid blurs away.
3. `normal_emb` (line 146): a **zero-initialised** linear branch for surface normals. Zero-init so it
   starts as a no-op (the model learns xyz first); normals get their own branch because pushing unit
   vectors through the sinusoidal high frequencies turned them into aliasing noise in the past.

**Training (separate, offline).** Supervised: `uv_mapper_loss` (line 193) = CE on u-bins + CE on
v-bins vs ground-truth UV from the sim mesh. **At runtime it is frozen**: `eval()`,
`requires_grad_(False)`. The policy never updates it.

**Where it runs.** `workers/uv_infer.py` (UV-only, old Haiku path) and inline inside
`workers/student_infer.py:55` (the StudentVLA path re-runs `uv_mapper.encode` itself). Same weights.

---

## 4. CORE FILE 2 — `model/student_vla.py` (the head, learned)

One shared point-cloud **transformer encoder** with **three read-outs**: grasp, drag, value
(actor-critic). `~20M params`. `d_model=384`, `n_enc=6`, `n_dec=4`, `max_wp=8`.

**Input** (`il_dataset.featurize`): `(B, N, 9) = [x, y, z, u, v, nx, ny, nz, z_abs]`. Columns 3–4 are
the **UV Mapper's output, baked in as input features** — this is the perception→policy join.

### 4.1 Tokenize + encode (`__init__:41`, `encode:66`)
```python
h = self.embed(x) + self.pos_mlp(x[..., :3])   # 9→384 per point + positional emb from xyz
H = self.encoder(h)                            # 6 self-attention layers → (B, N, 384)
```
Every point is a token; the encoder gives **one context-aware feature per point**, no pooling.
**No pooling is the whole point** of the rewrite: the old PointNet squashed all points to one 256-d
vector and lost *which point is where* (it was "too dumb to learn geometry"). Here nothing is
squashed. `dropout=0.0` so a re-forward reproduces a stored action's log-prob exactly (PPO needs
this).

### 4.2 GRASP head — "which point to grab" (`grasp_head:49`, `encode:70`)
```python
grasp_logits = self.grasp_head(H).squeeze(-1)  # (B, N) one logit per point → Categorical over N pts
```
The action is a **point index** `grab_idx ∈ [0, N-1]`, not an xyz. Why an index:
- always a **real, visible** point (no UV→point lookup that can miss),
- **permutation-equivariant / frame-free** → transfers to a real RealSense cloud with arbitrary order,
- **multimodal** → two good grab spots become two peaks; you sample one.

### 4.3 DRAG head — "where to move it" (`decode:72`) — a DETR-style decoder
```python
grab_feat = H[arange(B), grab_idx]             # ENCODER feature of the chosen point  (B, 384)
tgt = self.wp_queries + grab_feat.unsqueeze(1) # 8 learned query slots, each seeded by what we grabbed
dec = self.decoder(tgt, H)                     # queries CROSS-ATTEND all points  (B, 8, 384)
wp_mean  = self.wp_mean_head(dec)              # (B, 8, 3)  Gaussian mean per waypoint
stop_log = self.wp_stop_head(dec)             # (B, 8)     "is this waypoint real" logit
```
The drag head is the only actual transformer **decoder**. Critically, it does **not** take the grasp
head's *output* — it takes `H[grab_idx]`, the **encoder feature of the grasped point** (link is an
*index*, not a tensor). So the motion is conditioned on *what* you grabbed (sleeve vs hem → different
drag). Distributions (`sample:103`):
- `Normal(wp_mean, exp(wp_log_std))` for positions (`wp_log_std`, line 58 = learned exploration spread),
- `Bernoulli(stop_log)` for which waypoints are active → **variable-length path** (this is what makes
  1 Hz recording work — a long drag just turns on more waypoints).

`traj_split` (line 144): the **contiguous active prefix** is the trajectory; **last active = release**,
**the rest = path**. Frame: waypoint XY is centroid-relative, Z is absolute (table=0).

### 4.4 VALUE head — the critic (`value_head:62`, `value:121`)
```python
V = self.value_head(H.mean(1)).squeeze(-1)     # mean-pool → MLP → ONE scalar per state
```
The *only* place pooling is allowed — it produces a number, not an action, so per-point detail is
moot. `V(s)` = "expected discounted flatness-return from here." **Inference never uses it**; it exists
only as the PPO baseline.

### 4.5 The three call modes (same net, three lifecycle phases)
| Method | When | Returns |
|--------|------|---------|
| `forward(x, grab_idx)` (81) | **BC** | grasp_logits, wp_mean, stop_logits (grasp teacher-forced to demo) |
| `sample(x, greedy)` (89) | **rollout / eval** | action dict + `log_prob` (greedy=argmax for eval) |
| `evaluate(x, …)` (126) | **PPO** | (log_prob, entropy, value) from ONE encode |

### 4.6 Shape trace (one inference)
```
(4096,9) → embed+pos → encoder×6 → H(4096,384)
  grasp_head → (4096,) → Categorical → grab_idx=2317
  H[2317]=grab_feat → 8 queries+grab_feat → decoder(·,H) → (8,384)
      wp_mean (8,3) → Normal → waypoints
      wp_stop (8,)  → Bernoulli → active [1,1,1,0,0,0,0,0]
  traj_split → release=wp[2], path=[wp[0],wp[1]]
```

---

## 5. Data formats (the contracts between files)

**State `.npz`** (written by `_capture_rl_state`, head_RL.py:393):
`pcd_xyz (N,3) centroid-relative · normals (N,3) · centroid (3,) · pcd_to_mesh (N,)` (point→mesh
vertex index, needed to pin the grab patch at execution).

**Policy input `(N,9)`** (`il_dataset.featurize:157`): `[xyz, u, v, normals, z_abs]`, `z_abs =
pcd_z + centroid_z`. `u,v` come from the UV Mapper; everything else from the capture.

**Action dict** (out of `student_infer.py`): `grab_idx`, `grab_u/v` (=uv at that point), `release
[x,y,z]`, `path [[x,y,z]...]`, raw `waypoints (8,3)`, `active (8,)`, `log_prob`, `uv_pred_path`.

**BC label** (`il_dataset.action_to_targets:168`): `grab_idx` (point index, resolved via
`grab_pcd_idx` → nearest `grab_xyz` → nearest UV), `waypoints (8,3)`, `active (8,)`, `length k`.

**Recorded demo path** (`il_dataset.make_arm` → `_clean_path:65`): **variable-length, 1 Hz, not
padded**, capped to `MAX_WP-1` so path+release ≤ `MAX_WP`. Empty path = direct grab→release.

**RL buffer entry** (one per grab, head_RL `--rl-student` loop): adds `reward` (=ΔΦ this step),
`traj` (trajectory id), `t` (step), `done`, plus the action + `state_npz`/`uv_pred_path` so
`student_update` can re-featurize without the UV Mapper.

---

## 6. Training lifecycle

### 6.1 Collect (VR teleop, `infer`/sim, future)
Human smooths cloth in Isaac via OpenXR; each *grab* logs one `(state, action)` sample
(`il_dataset.record_sample` → `samples/*.npz` + `index.jsonl`). Target ~2k+, mild→extreme crumple.

### 6.2 BC pretrain — `workers/train_student.py`
```
L = grasp_w·CE(grasp_logits, grabbed_point)        # which point
  + drag_w ·MSE(waypoints, demo_path)·active_mask  # the trajectory (masked to real waypoints)
  + stop_w ·BCE(stop_logits, active)               # how many waypoints
```
→ `checkpoints/student_vla.pth`.

### 6.3 RL fine-tune — `head_RL.py --rl-student` + `workers/student_update.py` (multi-step PPO+GAE)
A grab is credited by the **discounted future of the whole smoothing sequence**, so a setup move that
lowers flatness now but unlocks a bigger gain later is rewarded.

**Reward** (`_flat_reward`, head_RL.py:850), pose-invariant:
```
Φ(s) = -(1.0·shape + 0.5·height) + 0.5·iou      # shape=Kabsch-aligned per-vertex XY dist to flat
r_t  = Φ(s_{t+1}) - Φ(s_t)                       # per-grab flatness IMPROVEMENT
```
(`Φ` uses ground-truth mesh verts → sim-only privilege in the *reward*, never a policy input.)

**Advantage/return — GAE** (`student_update.py`, per trajectory, reverse scan):
```
δ_t = r_t + γ·V(s_{t+1})·(1−done) − V(s_t)
Â_t = δ_t + γλ·(1−done)·Â_{t+1}                   # γ=0.97 λ=0.95
G_t = Â_t + V(s_t)                               # critic target
```
**Loss — clipped PPO** (`--rl-epochs` passes):
```
ρ   = exp(logπ_new − logπ_old)                   # old = behaviour log_prob in the buffer
L   = −mean(min(ρ·Â, clip(ρ,1±ε)·Â)) + 0.5·(V−G)² − 1e-3·entropy   # ε=0.2
```
Rollout: per crumple, branch `--rl-group-k` sequences from one snapshot, each `--rl-turns` grabs;
update every `--rl-k` episodes. See `STUDENT_RL.md` for the full knob table.

### 6.4 Eval — `head_RL.py --student-eval --gui`
Greedy (argmax grasp, mean drag), sequential turns; watch `rl_flat_overlay.png` converge.

---

## 7. Plumbing reference (every file, one line)

| File | Role |
|------|------|
| `model/uv_mapper.py` | **CORE** perception net (frozen) |
| `model/student_vla.py` | **CORE** policy+critic head (learned) |
| `il_dataset.py` | state/action formats: `featurize`, `action_to_targets`, `make_arm`, `record_sample` |
| `head_RL.py` | sim driver: capture, crumple, execute, reward, all the loops, arg parsing |
| `workers/uv_infer.py` | run UV Mapper → save `(N,2)` UV (old Haiku path) |
| `workers/student_infer.py` | UV Mapper encode + `policy.sample()` → action JSON |
| `workers/student_update.py` | PPO+GAE update of the head |
| `workers/train_student.py` | BC pretrain of the head |
| `workers/bc_teacher.py` | (teacher data helper) |
| `grasp_regions.py`, `data/label_grasp_regions.py` | deterministic UV-teacher regions (the `--scripted` path; superseded by the learned head) |
| `docs/HEAD.md` | deep dive: the deterministic UV teacher |
| `docs/STUDENT_RL.md` | deep dive: the head + multi-step PPO+GAE (source of truth for RL) |

---

## 8. "I want to change X" → edit Y (cheat-sheet)

| Want to change | Edit |
|----------------|------|
| Reward (what "flat" means / weights) | `head_RL.py:_flat_reward` (`FLAT_W_SHAPE/FLAT/COV`, ~line 788) |
| RL knobs (γ, λ, clip, epochs, entropy, turns, branches) | `head_RL.py` arg defaults ~line 56–73; passed to `student_update.py` |
| Head size (layers, d_model, heads, max waypoints) | `model/student_vla.py:__init__` (`d_model, n_enc, n_dec, n_heads, max_wp`) |
| Max trajectory length | `il_dataset.MAX_WP` (line 54) — **set before BC** (baked into checkpoint) |
| Grasp representation (point index vs UV) | `model/student_vla.py` grasp_head + `head_RL.py:_student_arm` |
| Point count (downsample) | `head_RL.py:N_PCD` (line 113, =4096) |
| UV bins / UV Mapper depth | `model/uv_mapper.py:K` (line 21), `UVMapper.__init__` |
| BC loss weights | `workers/train_student.py` (`--grasp-weight/--drag-weight/--stop-weight`) |
| Drag speed / safety clamps | `head_RL.py:--drag-speed` (line 73), `_clamp_wp` (line 905) |
| How demos are recorded | `il_dataset.py:make_arm` / `_clean_path` |
| Crumple difficulty | `head_RL.py:_crumple` (line 1122) |

---

## 9. Gotchas / known rough edges (don't get bitten)

- **Checkpoint name drift**: `workers/uv_infer.py:25` defaults to `uv_mapper2_best.pth`, but
  `head_RL.py` and `student_infer.py` pass/default to `uv_mapper_best.pth`. The file actually loaded
  is whatever `--model` passes. Reconcile these so you don't train one and load the other.
- **Old checkpoints are incompatible** with the current head (PointNet→transformer→+critic rewrite).
  Retrain BC, then RL. `student_infer`/`student_update` load with `strict=False` to tolerate a
  pre-critic BC checkpoint (the value head just starts random and PPO trains it).
- **`MAX_WP` is baked into the checkpoint.** Changing it requires re-BC. A VR drag longer than
  `MAX_WP-1` path points (~7 s @ 1 Hz) gets truncated — raise `MAX_WP` first if your pulls run long.
- **The reward uses ground-truth mesh verts** (`verts()`), a sim privilege. Fine for sim RL; it is
  never a policy input, so nothing leaks into real-world inference. There is no equivalent reward on
  a real robot (you'd need a different success signal).
- **`dropout=0.0` in the head is load-bearing**, not a tuning choice — it makes PPO's log-prob recompute exact.

---

## 10. Glossary

- **UV** — coordinates `(u,v) ∈ [0,1]²` on the garment's flat 2D pattern; the "identity" of a point.
- **BC** — Behavioural Cloning: supervised imitation of demos (no reward).
- **RL** — Reinforcement Learning: improve via a reward signal.
- **PPO** — Proximal Policy Optimization: policy-gradient update that *clips* the probability ratio so
  one step can't move the policy too far (lets you do several epochs per batch safely).
- **GAE** — Generalized Advantage Estimation: low-variance estimate of "how much better than expected"
  an action was, blending short- and long-horizon returns with λ.
- **Advantage `Â`** — action return minus the critic's expectation; `>0` ⇒ reinforce it.
- **Critic / `V(s)`** — predicted expected return from a state; the PPO baseline.
- **γ (gamma)** — discount; how far into the future credit reaches.
- **Categorical / Normal / Bernoulli** — the three action distributions (grasp / waypoint xyz / stop).
- **FPS** — Furthest Point Sampling: downsample the cloud to N=4096 spread-out points.
- **Kabsch** — closed-form best-fit rigid alignment; used to compare shape pose-invariantly in `Φ`.
- **DETR-style queries** — a small fixed set of learned tokens that cross-attend a set to read out a
  fixed number of structured outputs (here, the 8 waypoint slots).
- **spconv** — sparse 3D convolution (compute only on occupied voxels) used in the UV Mapper.